# Mac: lokal OOF inference + SDF cache

Bu notebook TrackB öncesindeki bir defalık Adım 4–5'i Mac'te çalıştırır. Dataset, checkpoint ve cache **lokal SSD** üzerinde kalır. Drive'a büyük dosya aktarımı en sonda, ayrı ve isteğe bağlı hücrededir.

## 0. Bir defalık ortam

Yeni bir Python environment kullanıyorsan bu hücre internet indirir; mobil veri kullanırken çalıştırma. Paketler zaten kuruluysa atla.

In [1]:
# Gerekirse bir kez çalıştır:
%pip install nnunetv2 nibabel scipy scikit-image pyyaml torch matplotlib
import torch
print('MPS available:', torch.backends.mps.is_available())
assert torch.backends.mps.is_available(), 'Apple Silicon MPS backend bulunamadı'


  Using cached nnunetv2-2.8.1-py3-none-any.whl.metadata (18 kB)
  Using cached acvl_utils-0.2.6.tar.gz (34 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached dynamic_network_architectures-0.4.4-py3-none-any.whl.metadata (13 kB)
  Using cached batchgenerators-0.25.3-py3-none-any.whl.metadata (22 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached imagecodecs-2026.6.26-cp312-abi3-macosx_11_0_arm64.whl.metadata (23 kB)
  Using cached yacs-0.1.8-py3-none-any.whl.metadata (639 bytes)
  Using cached batchgeneratorsv2-0.3.4-py3-none-any.whl.metadata (15 kB)
  Using cached connected_components_3d-4.0.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (33 kB)
  Using cached timm-1.0.22-py3-none-any.whl.metadata (63 kB)
  Using cached future-1.0.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached unittest2-1.1.0-py2.py3-none-any.whl.metadata (15 kB)
  Using cached ndinde

## 1. Lokal yollar ve checkpoint kontrolü

Önce indirdiğin üç checkpoint zip'ini aynı `~/Downloads/iac_merged` klasörüne aç. Bu hücre beş `checkpoint_final.pth` dosyasını bulamazsa durur.

In [2]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

REPO = Path('/Users/anil/Desktop/GitHub/ToothFairy3-IAC-Segmentation-Flow')
GDRIVE = Path("/Users/anil/Library/CloudStorage/GoogleDrive-anil04keskin@gmail.com/Drive'ım")
DRIVE_RUNS = GDRIVE / 'ToothFairy/ToothFairy3/iac_runs'
DATASET = Path('/Users/anil/tf_work/Dataset801_IAC_LR')
LOCAL_RUNS = Path('/Users/anil/tf_work/iac_runs')
CHECKPOINT_RESULTS = Path('/Users/anil/Downloads/iac_merged/iac_runs/nnUNet_results')
TRAINER = 'nnUNetTrainerIAC_NoMirror'

IMAGES, LABELS = DATASET / 'imagesTr', DATASET / 'labelsTr'
SPLITS = REPO / 'configs/splits.json'
assert REPO.is_dir(), REPO
assert IMAGES.is_dir() and LABELS.is_dir(), DATASET
assert SPLITS.is_file(), f'Missing split: {SPLITS}'
assert CHECKPOINT_RESULTS.is_dir(), (
    f'Checkpoint arşivlerini birleştir: {CHECKPOINT_RESULTS}')

# nnU-Net CLI bu değişkenleri bekler. Raw/preprocessed inference sırasında boş olabilir.
NNUNET_WORK = Path('/Users/anil/tf_work/nnunet_runtime')
os.environ['nnUNet_raw'] = str(NNUNET_WORK / 'raw')
os.environ['nnUNet_preprocessed'] = str(NNUNET_WORK / 'preprocessed')
os.environ['nnUNet_results'] = str(CHECKPOINT_RESULTS)
os.environ['nnUNet_extTrainer'] = str(REPO / 'nnunet')
for key in ('nnUNet_raw', 'nnUNet_preprocessed'):
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)

results_root = CHECKPOINT_RESULTS / 'Dataset801_IAC_LR' / f'{TRAINER}__nnUNetPlans__3d_fullres'
for fold in range(5):
    checkpoint = results_root / f'fold_{fold}/checkpoint_final.pth'
    assert checkpoint.is_file(), f'Missing: {checkpoint}'
print('Checkpoint root:', results_root)
print('External trainers:', os.environ['nnUNet_extTrainer'])
print('Local dataset:', DATASET)


Checkpoint root: /Users/anil/Downloads/iac_merged/iac_runs/nnUNet_results/Dataset801_IAC_LR/nnUNetTrainerIAC_NoMirror__nnUNetPlans__3d_fullres
Local dataset: /Users/anil/tf_work/Dataset801_IAC_LR


## 2. Split doğrulaması

TrackA checkpoint'lerinin eğitildiği fold atamasıyla bu dosya aynı olmalıdır. Drive'da küçük split yedeği varsa byte-byte eşitlik ayrıca kontrol edilir.

In [3]:
drive_split = DRIVE_RUNS / 'configs_cache/splits.json'
if drive_split.is_file():
    assert SPLITS.read_bytes() == drive_split.read_bytes(), 'Repo ve Drive splits farklı: dur'
print('split sha256:', hashlib.sha256(SPLITS.read_bytes()).hexdigest())
split_data = json.loads(SPLITS.read_text())
print('development:', len(split_data['development']), '| folds:', len(split_data['folds']))


split sha256: 85a8b74f947c8cd84a22a84afdece98ae6c8a6d48c695254a0cf8623a1c88b79
development: 480 | folds: 5


## 3. OOF prediction — lokal MPS

Bu işlem beş fold ile toplam 480 development vakasını tahmin eder. `--resume` yarım kalırsa geçerli cache'leri atlar.

In [ ]:
LOCAL_CACHE = LOCAL_RUNS / 'sdf_cache'
OOF_OUT = LOCAL_CACHE / 'oof_probs'
GT_SDF_OUT = LOCAL_CACHE / 'gt_sdf'
COARSE_SDF_OUT = LOCAL_CACHE / 'coarse_sdf'
for path in (OOF_OUT, GT_SDF_OUT, COARSE_SDF_OUT):
    path.mkdir(parents=True, exist_ok=True)
os.chdir(REPO)

subprocess.run([
    sys.executable, 'nnunet/predict_oof.py',
    '--dataset', '801', '--config', '3d_fullres',
    '--trainer', TRAINER, '--splits', str(SPLITS),
    '--images', str(IMAGES), '--out', str(OOF_OUT),
    '--device', 'mps', '--step-size', '0.5', '--resume',
], check=True)


[oof] nnUNetv2_predict -i /var/folders/4g/l3t8zydd6r13p1v7608kyh8h0000gn/T/tmpi44r0txo -o /var/folders/4g/l3t8zydd6r13p1v7608kyh8h0000gn/T/tmprbfqdvkx -d 801 -c 3d_fullres -tr nnUNetTrainerIAC_NoMirror -f 0 --disable_tta -device mps -step_size 0.5 -npp 3 -nps 3 --not_on_device

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

perform_everything_on_device=True is only supported for cuda devices! Setting this to False
Trainer 'nnUNetTrainerIAC_NoMirror' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /Users/anil/Desktop/GitHub/ToothFairy3-IAC-Seg

100%|██████████| 120/120 [03:46<00:00,  1.89s/it]


sending off prediction to background worker for resampling and export
done with ToothFairy3F_007

Predicting ToothFairy3F_009:
perform_everything_on_device: False


100%|██████████| 120/120 [03:42<00:00,  1.85s/it]


sending off prediction to background worker for resampling and export
done with ToothFairy3F_009

Predicting ToothFairy3F_010:
perform_everything_on_device: False


...

100%|██████████| 120/120 [03:46<00:00,  1.89s/it]


sending off prediction to background worker for resampling and export
done with ToothFairy3F_022

Predicting ToothFairy3F_023:
perform_everything_on_device: False


100%|██████████| 120/120 [03:46<00:00,  1.89s/it]


sending off prediction to background worker for resampling and export
done with ToothFairy3F_023

Predicting ToothFairy3F_025:
perform_everything_on_device: False


 99%|█████████▉| 119/120 [03:37<00:01,  1.81s/it]

## 4. SDF cache — lokal CPU

In [ ]:
subprocess.run([
    sys.executable, 'data/compute_gt_sdf.py',
    '--labels', str(LABELS), '--out', str(GT_SDF_OUT),
    '--clip-mm', '10', '--resume',
], check=True)
subprocess.run([
    sys.executable, 'data/compute_coarse_sdf.py',
    '--oof', str(OOF_OUT), '--ref-labels', str(LABELS),
    '--out', str(COARSE_SDF_OUT), '--prob-thresh', '0.5',
    '--clip-mm', '10', '--resume',
], check=True)


## 5. Doğrula

Bu hücre hatasız bitmeden Drive'a aktarım veya Colab'a geçme.

In [ ]:
subprocess.run([
    sys.executable, 'data/validate_pipeline_cache.py',
    '--splits', str(SPLITS), '--images', str(IMAGES),
    '--labels', str(LABELS), '--oof', str(OOF_OUT),
    '--gt-sdf', str(GT_SDF_OUT), '--coarse-sdf', str(COARSE_SDF_OUT),
], check=True)
print('READY FOR DRIVE SYNC:', LOCAL_CACHE)


## 6. İsteğe bağlı: Wi‑Fi varken Drive'a aktar

Bu hücre büyük upload yapar. Mobil veriyle çalıştırma. Colab TrackB için dataset ve üç cache klasörü Drive'da gerekli olacaktır.

In [ ]:
SYNC_INPUTS_TO_DRIVE = False  # Dataset + checkpoint için Wi-Fi'da True yap
SYNC_CACHES_TO_DRIVE = False  # OOF/SDF bittikten sonra Wi-Fi'da True yap

def copy_tree(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print('synced:', destination)

if SYNC_INPUTS_TO_DRIVE:
    copy_tree(DATASET, DRIVE_RUNS / 'dataset_cache/Dataset801_IAC_LR')
    copy_tree(CHECKPOINT_RESULTS, DRIVE_RUNS / 'nnUNet_results')
    drive_split.parent.mkdir(parents=True, exist_ok=True)
    drive_split.write_bytes(SPLITS.read_bytes())
    print('Dataset/checkpoint upload başlatıldı; Drive Desktop senkronunu tamamlamalı.')
if SYNC_CACHES_TO_DRIVE:
    copy_tree(OOF_OUT, DRIVE_RUNS / 'sdf_cache_backup/oof_probs')
    copy_tree(GT_SDF_OUT, DRIVE_RUNS / 'sdf_cache_backup/gt_sdf')
    copy_tree(COARSE_SDF_OUT, DRIVE_RUNS / 'sdf_cache_backup/coarse_sdf')
    print('OOF/SDF cache upload başlatıldı; Drive Desktop senkronunu tamamlamalı.')
if not (SYNC_INPUTS_TO_DRIVE or SYNC_CACHES_TO_DRIVE):
    print('Yerel çıktı korundu. Wi-Fi hazır olduğunda ilgili SYNC bayrağını True yap.')
